# Preprocessing – Book Dataset (Tiki)

Mục tiêu:
- Loại bỏ cột không cần thiết (`cat_level_1`, `cat_level_5`)
- Đổi tên cột danh mục: `cat_level_2→cat1`, `cat_level_3→cat2`, `cat_level_4→cat3`
- Xử lý missing value
- Lọc chỉ giữ sách có `review_count > 0`
- Lưu vào `data/processed/book_dataset_clean.csv`

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH  = Path('../data/raw/book_dataset.csv')
OUT_PATH  = Path('../data/processed/book_dataset_clean.csv')

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df.shape}')
df.head(3)

## 1. Tổng quan dữ liệu

In [ ]:
print('=== Kiểu dữ liệu ===')
print(df.dtypes)
print()
print('=== Missing values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(summary[summary['missing'] > 0].to_string())

In [ ]:
print('cat_level_1 unique:', df['cat_level_1'].unique().tolist())
print('cat_level_2 unique:', df['cat_level_2'].unique().tolist())
print('cat_level_3 nunique:', df['cat_level_3'].nunique())
print('cat_level_4 nunique:', df['cat_level_4'].nunique())
print('cat_level_5 nunique:', df['cat_level_5'].nunique())

## 2. Loại bỏ cột không cần thiết

- `cat_level_1`: chỉ có 1 giá trị duy nhất (`nha-sach-tiki`) → không có thông tin phân biệt
- `cat_level_5`: 63.5% missing, quá ít dữ liệu để dùng

In [ ]:
df = df.drop(columns=['cat_level_1', 'cat_level_5'])
print(f'Shape sau khi bỏ cột: {df.shape}')
print('Columns:', df.columns.tolist())

## 3. Đổi tên cột danh mục

In [ ]:
df = df.rename(columns={
    'cat_level_2': 'cat1',
    'cat_level_3': 'cat2',
    'cat_level_4': 'cat3',
})

print('cat1 (ngôn ngữ):', df['cat1'].unique().tolist())
print('cat2 (thể loại):', df['cat2'].nunique(), 'giá trị')
print('cat3 (chủ đề chi tiết):', df['cat3'].nunique(), 'giá trị')

## 4. Xử lý Missing Value

| Cột | Tỷ lệ missing | Xử lý |
|-----|--------------|--------|
| `authors` | 25.2% | Fill `"Không rõ"` |
| `publisher_vn` | 2.2% | Fill `"Không rõ"` |
| `manufacturer` | 6.0% | Fill `"Không rõ"` |
| `current_seller_name` | 1.3% | Fill `"Không rõ"` |
| `stock` | 1.3% | Fill `0` |
| `cat3` | 0.2% | Fill `"Không rõ"` |
| `isbn13` | 43.2% | Giữ nguyên (không dùng trong phân tích) |
| `book_cover` | 29.0% | Giữ nguyên (metadata ảnh) |
| `number_of_page` | 63.9% | Giữ nguyên (quá nhiều missing) |
| `publication_date` | 53.5% | Giữ nguyên (quá nhiều missing) |

In [ ]:
# Fill text columns với "Không rõ"
text_fill = ['authors', 'publisher_vn', 'manufacturer', 'current_seller_name', 'cat3']
for col in text_fill:
    before = df[col].isnull().sum()
    df[col] = df[col].fillna('Không rõ')
    print(f'{col}: filled {before} missing values')

# Fill stock với 0
before = df['stock'].isnull().sum()
df['stock'] = df['stock'].fillna(0)
print(f'stock: filled {before} missing values with 0')

In [ ]:
print('=== Missing values sau xử lý ===')
remaining = df.isnull().sum()
remaining_pct = (remaining / len(df) * 100).round(1)
summary2 = pd.DataFrame({'missing': remaining, 'pct': remaining_pct})
print(summary2[summary2['missing'] > 0].to_string())

## 5. Lọc sách có review_count > 0

Chỉ giữ sách đã có ít nhất 1 lượt đánh giá — đây là sách đã từng được mua và đánh giá, có dữ liệu thực tế.

In [ ]:
before = len(df)
df = df[df['review_count'] > 0].reset_index(drop=True)
after = len(df)

print(f'Trước lọc : {before:,} dòng')
print(f'Sau lọc   : {after:,} dòng')
print(f'Loại bỏ   : {before - after:,} dòng ({(before - after) / before * 100:.1f}%)')

## 6. Kiểm tra cuối & Lưu file

In [ ]:
print(f'Final shape: {df.shape}')
print()
print('=== Thống kê mô tả ===')
df[['price', 'list_price', 'discount_rate', 'rating_average', 'review_count', 'all_time_quantity_sold']].describe().round(2)

In [ ]:
print('cat1 phân bố:')
print(df['cat1'].value_counts())
print()
print('cat2 top 10:')
print(df['cat2'].value_counts().head(10))

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f'Đã lưu: {OUT_PATH}')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')